In [1]:
import os

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable, RunnableParallel
from langchain_openai import ChatOpenAI

In [2]:
# 使用小米 MiMo 模型

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0.7,
)

In [3]:
# --- 定义三个独立的链 ---
# 这三个链代表可以并行执行的不同任务

# 1. 总结链：生成主题摘要
summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages([("system", "简洁地总结以下主题:"), ("user", "{topic}")]) | llm | StrOutputParser()
)

# 2. 问题链：生成相关问题
questions_chain: Runnable = (
    ChatPromptTemplate.from_messages([("system", "生成三个关于以下主题的有趣问题:"), ("user", "{topic}")])
    | llm
    | StrOutputParser()
)

# 3. 关键词链：提取关键术语
terms_chain: Runnable = (
    ChatPromptTemplate.from_messages([("system", "从以下主题中识别5-10个关键术语，用逗号分隔:"), ("user", "{topic}")])
    | llm
    | StrOutputParser()
)


In [6]:
# --- 构建并行 + 合成链 ---

# 1. 定义并行执行的任务块
map_chain = RunnableParallel({"summary": summarize_chain, "questions": questions_chain, "terms": terms_chain})

# 2. 定义合成提示词
synthesis_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """根据以下信息，生成一个简单的回答：
     摘要：{summary}
     问题：{questions}
     关键术语：{terms}""",
        ),
        ("user", "请综合以上内容，提供一个完整的分析."),
    ]
)

# 3. 组合完整链：并行执行 → 合成结果
full_chain = map_chain | synthesis_prompt | llm | StrOutputParser()


In [7]:
# --- 运行示例 ---
topic = "AI Agent 设计模式"

print("--- 并行执行三个任务 ---")
result = full_chain.invoke({"topic": topic})
print("\n--- 最终结果 ---")
print(result)


--- 并行执行三个任务 ---

--- 最终结果 ---
基于您提供的材料，我将对 **AI Agent 设计模式** 进行一个综合性的分析，整合其核心概念、模式对比，并深入探讨您提出的三个关键问题。

### **一、 核心概念与模式全景**

AI Agent 的本质是 **“感知-决策-行动”** 的循环系统。其设计模式的核心在于如何编排 **LLM（推理引擎）**、**记忆**、**工具** 和 **规划策略** 这四大组件，以完成复杂任务。

您列出的七种模式可归纳为三个层次：
1.  **基础推理模式**：**ReAct**（推理与行动交替）是基石，提供了最基本的“思考-行动”循环。
2.  **规划与反思层**：
    *   **Plan-and-Execute**：强调“先谋后动”，适合明确任务。
    *   **Reflection**：引入“自我评估与修正”，提升输出质量。
    *   **Tree of Thoughts**：通过探索多条路径进行“深度推理”。
3.  **协作与扩展层**：
    *   **Multi-Agent**：通过专业化分工应对复杂性。
    *   **Tool Use**：通过外部工具突破能力边界。
    *   **Chaining**：通过串联实现流程化。

### **二、 对三个核心挑战的深度分析**

这三个问题分别触及了 Agent 设计的**认知边界、架构权衡与环境适应性**，是构建稳健系统的关键。

#### **1. 反思的终止：避免“分析瘫痪”与设计最优深度**
**问题本质**：这是关于 **Agent 的元认知与效率平衡** 的问题。

*   **类比人类**：这与人类的“过度思考”高度相似。无限反思会导致决策延迟（分析瘫痪）和计算资源浪费。
*   **终止条件设计**：
    *   **基于效用/成本**：设定一个“反思收益阈值”。如果新一轮反思带来的预期改进小于其消耗的计算成本或时间，则终止。
    *   **基于置信度**：当 Agent 对当前结果的“置信度”超过某个阈值时停止。这需要设计可靠的置信度评估机制。
    *   **基于资源限制**：设置最大反思轮次或总时间，作为硬性约束。
    *   **基于外部验证**：引入一个轻量级的“验